# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [26]:
import pandas as pd
import numpy as np
import os
import shutil
import fasttext
import fasttext.util
from tqdm import tqdm

import warnings
warnings.filterwarnings(action='ignore')

## 1.2 Функции

In [8]:
def download_fasttext_pretrained(language='ru', download_dir='./fasttext_models'):
    """
    Скачивает предобученные модели через fasttext библиотеку
    """
    os.makedirs(download_dir, exist_ok=True)
    
    model_filename = f'cc.{language}.300.bin'
    model_path = os.path.join(download_dir, model_filename)
    
    if os.path.exists(model_path):
        print(f"Модель уже существует: {model_path}")
        return model_path
    
    print(f"Скачивание FastText модели для языка {language}...")
    
    try:
        # Скачиваем модель
        fasttext.util.download_model(language, if_exists='ignore')
        
        # Перемещаем в нужную директорию
        original_path = f'cc.{language}.300.bin'
        if os.path.exists(original_path):
            shutil.move(original_path, model_path)
            print(f"Модель сохранена в: {model_path}")
            return model_path
        else:
            print("Не удалось найти скачанную модель")
            return None
            
    except Exception as e:
        print(f"Ошибка при скачивании: {e}")
        return None

In [18]:
def vectorize_headings_fasttext(df, text_columns=None, model_path=None, pooling='mean', **fasttext_params):
    """
    Векторизует текстовые колонки с помощью предобученной FastText модели
    
    Parameters:
    df - исходный DataFrame
    text_columns - список колонок для обработки
    model_path - путь к предобученной FastText модели
    pooling - метод агрегации векторов слов ('mean', 'sum', 'max')
    fasttext_params - дополнительные параметры
    """
    
    if text_columns is None:
        text_columns = [col for col in df.columns if col.startswith('heading_')]
    
    if model_path is None:
        raise ValueError("Не указан путь к FastText модели")
    
    # Загружаем модель
    print(f"Загрузка FastText модели из: {model_path}")
    try:
        model = fasttext.load_model(model_path)
        print("Модель успешно загружена")
    except Exception as e:
        print(f"Ошибка при загрузке модели: {e}")
        return df
    
    # Создаем копию DataFrame без текстовых колонок
    df_vectorized = df.drop(columns=text_columns, errors='ignore').copy()
    
    # Параметры по умолчанию
    default_params = {
        'pooling': pooling,
        'normalize': True  # нормализация векторов
    }
    default_params.update(fasttext_params)
    
    def text_to_vector(text, model, pooling_method='mean'):
        """Преобразует текст в вектор с помощью FastText"""
        if pd.isna(text) or text == '':
            return np.zeros(model.get_dimension())
        
        # Токенизация (простая разбивка по пробелам)
        words = str(text).split()
        
        if not words:
            return np.zeros(model.get_dimension())
        
        # Получаем векторы для каждого слова
        word_vectors = []
        for word in words:
            try:
                word_vector = model.get_word_vector(word)
                word_vectors.append(word_vector)
            except:
                continue
        
        if not word_vectors:
            return np.zeros(model.get_dimension())
        
        word_vectors = np.array(word_vectors)
        
        # Применяем выбранный метод агрегации
        if pooling_method == 'mean':
            text_vector = np.mean(word_vectors, axis=0)
        elif pooling_method == 'sum':
            text_vector = np.sum(word_vectors, axis=0)
        elif pooling_method == 'max':
            text_vector = np.max(word_vectors, axis=0)
        else:
            text_vector = np.mean(word_vectors, axis=0)
        
        # Нормализация вектора
        if default_params['normalize']:
            norm = np.linalg.norm(text_vector)
            if norm > 0:
                text_vector = text_vector / norm
        
        return text_vector
    
    for col in text_columns:
        if col not in df.columns:
            continue
            
        print(f"Векторизация колонки: {col}")
        
        # Заменяем NaN на пустые строки
        texts = df[col].fillna('')
        
        try:
            # Векторизуем все тексты
            vectors = []
            for text in tqdm(texts, desc=f"Обработка {col}"):
                vector = text_to_vector(text, model, default_params['pooling'])
                vectors.append(vector)
            
            vectors = np.array(vectors)
            
            # Создаем DataFrame с векторами
            feature_names = [f"{col}_fasttext_{i}" for i in range(vectors.shape[1])]
            fasttext_df = pd.DataFrame(vectors, columns=feature_names, index=df.index)
            
            # Добавляем к результату
            df_vectorized = pd.concat([df_vectorized, fasttext_df], axis=1)
            
            print(f"Создано {len(feature_names)} признаков для {col}")
            
        except Exception as e:
            print(f"Ошибка при векторизации {col}: {e}")
            # Если ошибка, создаем пустые колонки с правильной размерностью
            empty_vectors = np.zeros((len(df), model.get_dimension()))
            feature_names = [f"{col}_fasttext_{i}" for i in range(model.get_dimension())]
            empty_df = pd.DataFrame(empty_vectors, columns=feature_names, index=df.index)
            df_vectorized = pd.concat([df_vectorized, empty_df], axis=1)
    
    return df_vectorized

# 2 Чтение данных

In [14]:
data = pd.read_csv("../../data/news/3_titles_processed.csv")

In [16]:
data.head(2)

,begin,heading_interfax,heading_vedomosti,heading_kommersant
0,2022-05-01 10:00:00,володин предлож конфисковыва актив владельц би...,NaN,росс начина при заявлен нов выплат нужда дет с...
1,2022-05-01 12:00:00,мишустин подписа постановлен снижен ставк льго...,правительств одобр снижен ставк льготн ипотек ...,NaN


# 3 Обработка

In [20]:
model_path = download_fasttext_pretrained(
    language='ru', 
    download_dir='../../models/fasttext'
)

Модель уже существует: ../../models/fasttext\cc.ru.300.bin


In [28]:
df_vectorized = vectorize_headings_fasttext(
    df=data,
    text_columns=['heading_interfax', 'heading_vedomosti', 'heading_kommersant'],
    model_path=model_path,
    pooling='mean',
    normalize=True
)

Загрузка FastText модели из: ../../models/fasttext\cc.ru.300.bin
Модель успешно загружена
Векторизация колонки: heading_interfax


Обработка heading_interfax: 100%|████████████████████████████████████████████████| 6250/6250 [00:04<00:00, 1301.67it/s]


Создано 300 признаков для heading_interfax
Векторизация колонки: heading_vedomosti


Обработка heading_vedomosti: 100%|███████████████████████████████████████████████| 6250/6250 [00:02<00:00, 2144.79it/s]


Создано 300 признаков для heading_vedomosti
Векторизация колонки: heading_kommersant


Обработка heading_kommersant: 100%|██████████████████████████████████████████████| 6250/6250 [00:03<00:00, 2055.88it/s]


Создано 300 признаков для heading_kommersant


In [30]:
df_vectorized.head()

,begin,heading_interfax_fasttext_0,heading_interfax_fasttext_1,heading_interfax_fasttext_2,heading_interfax_fasttext_3,heading_interfax_fasttext_4,heading_interfax_fasttext_5,heading_interfax_fasttext_6,heading_interfax_fasttext_7,heading_interfax_fasttext_8,...,heading_kommersant_fasttext_290,heading_kommersant_fasttext_291,heading_kommersant_fasttext_292,heading_kommersant_fasttext_293,heading_kommersant_fasttext_294,heading_kommersant_fasttext_295,heading_kommersant_fasttext_296,heading_kommersant_fasttext_297,heading_kommersant_fasttext_298,heading_kommersant_fasttext_299
0,2022-05-01 10:00:00,0.124890,-0.101783,-0.017166,0.030840,0.006648,-0.104627,-0.015941,0.008354,0.027082,...,-0.046253,0.002917,0.016700,-0.066125,-0.008185,0.015039,0.094410,0.007845,-0.005977,-0.035276
1,2022-05-01 12:00:00,0.079931,-0.120844,0.019048,-0.042986,0.004966,-0.124700,-0.002416,-0.023473,0.068674,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2022-05-01 14:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.043052,-0.005205,0.007689,-0.043272,-0.088173,-0.037961,0.015981,0.032976,0.057851,-0.036435
3,2022-05-01 16:00:00,0.073224,-0.062680,0.028672,0.026615,0.039821,-0.047871,0.035894,0.018390,-0.037843,...,-0.037843,0.009130,0.001020,-0.056731,0.015740,-0.033612,0.106279,-0.022522,-0.017395,-0.024772
4,2022-05-01 18:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.043679,0.006082,0.000611,-0.053091,0.002115,-0.055400,0.151366,-0.006134,-0.024480,-0.028695


In [32]:
df_vectorized.to_csv("../../data/news/3_titles_fasttext.csv", index=False)